In [3]:
from pathlib import Path
from collections import Counter
import json
import math
import random
import shutil
import xml.etree.ElementTree as ET

import matplotlib.pylab as plt
from matplotlib.patches import Rectangle
import numpy as np
from PIL import Image

**configuration**

In [4]:
SEED = 42
CLASS_TO_ID = {
    "D00": 0,
    "D10": 1,
    "D20": 2,
    "D40": 3,
}

CLASS_NAMES = {
    "D00": "Longitudinal crack",
    "D10": "Transverse crack",
    "D20": "Alligator crack",
    "D40": "Pothole",
}
REVIEWED_IGNORED_CLASSES = set()
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png",}

In [6]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


RAW_DATASET_DIR = Path("/content/RDD2022_China_MotorBike/China_MotorBike")

TRAIN_IMAGES_DIR = (RAW_DATASET_DIR / "train" / "images")
TRAIN_XML_DIR = (RAW_DATASET_DIR / "train" / "annotations" / "xmls")
TEST_IMAGES_DIR = (RAW_DATASET_DIR / "test" / "images")


PROCESSED_ALL_DIR = (PROJECT_ROOT / "data" / "processed" / "all")
PROCESSED_IMAGES_DIR = (PROCESSED_ALL_DIR / "images")
PROCESSED_LABELS_DIR = (PROCESSED_ALL_DIR / "labels")


REPORTS_DIR = (PROJECT_ROOT / "reports")
FIGURES_DIR = (REPORTS_DIR / "figures")


print("Project root :", PROJECT_ROOT)
print("Dataset root:", RAW_DATASET_DIR)
print("Train images:", TRAIN_IMAGES_DIR)
print("Train XMLs  :", TRAIN_XML_DIR)
print("Test images :", TEST_IMAGES_DIR)

Project root : /content
Dataset root: /content/RDD2022_China_MotorBike/China_MotorBike
Train images: /content/RDD2022_China_MotorBike/China_MotorBike/train/images
Train XMLs  : /content/RDD2022_China_MotorBike/China_MotorBike/train/annotations/xmls
Test images : /content/RDD2022_China_MotorBike/China_MotorBike/test/images


**File inventory and pairing audit**

In [7]:
def collect_images(directory):
    return sorted(
        path for path in directory.iterdir() if (
            path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        )
    )

image_files = collect_images(TRAIN_IMAGES_DIR)
xml_files = sorted(TRAIN_XML_DIR.glob("*.xml"))
test_image_files = collect_images(TEST_IMAGES_DIR)

image_stems = [path.stem for path in image_files]
xml_stems = [path.stem for path in xml_files]

In [8]:
duplicate_image_stems = [
    stem for stem, count in Counter(image_stems).items() if count > 1
]
duplicate_xml_stems = [
    stem for stem, count in Counter(xml_stems).items() if count > 1
]

images_by_stem = {path.stem: path for path in image_files}
xml_by_stem = {path.stem: path for path in xml_files}

In [9]:
images_without_xml = sorted(set(images_by_stem) - set(xml_by_stem))

xml_without_images = sorted(set(xml_by_stem) - set(images_by_stem))

print("Train images       :", len(image_files))
print("Train XML files    :", len(xml_files))
print("Test images        :", len(test_image_files))
print("Images without XML :", len(images_without_xml))
print("XML without image  :", len(xml_without_images))
print("Duplicate images   :", len(duplicate_image_stems))
print("Duplicate XMLs     :", len(duplicate_xml_stems))

Train images       : 1977
Train XML files    : 1977
Test images        : 500
Images without XML : 0
XML without image  : 0
Duplicate images   : 0
Duplicate XMLs     : 0
